# Souhrnne RMS druzic - Hermite GOP vs SSA

Souhrnne RMS za cele zpracovane obdobi z jiz ulozenych dennich Hermite statistik. Hermitova interpolace se zde znovu nepocita.

In [7]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from doris.output.tables import Col, save_latex_table

RESULTS_DIR = PROJECT_ROOT / "data" / "results"
TABLES_DIR = PROJECT_ROOT / "LaTeX" / "tables" / "results" / "satellites" / "hermite"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RESULTS_DIR:  {RESULTS_DIR}")
print(f"TABLES_DIR:   {TABLES_DIR}")

PROJECT_ROOT: C:\Users\michal\Desktop\MasterThesis-DorisAnalysis
RESULTS_DIR:  C:\Users\michal\Desktop\MasterThesis-DorisAnalysis\data\results
TABLES_DIR:   C:\Users\michal\Desktop\MasterThesis-DorisAnalysis\LaTeX\tables\results\satellites\hermite


In [8]:
SATELLITES = {
    "srl": "SARAL",
    "cs2": "CryoSat-2",
    "h2c": "HY-2C",
    "h2d": "HY-2D",
    "ja3": "Jason-3",
    "s3a": "Sentinel-3A",
    "s3b": "Sentinel-3B",
    "s6a": "Sentinel-6A",
}

SOLUTION_TAG = "gop_vs_ssa"
EXPECTED_DAYS = 30

# Pri stejnem poctu epoch za den je sqrt(mean(daily_rms**2))
# totozne se souhrnnym RMS pres vsechny epochy v obdobi.
RMS_COLUMNS = {
    "R_rms": "R_rms_mm",
    "T_rms": "T_rms_mm",
    "N_rms": "N_rms_mm",
}

In [9]:
def aggregate_rms_from_daily_rms(series: pd.Series) -> float:
    values = pd.to_numeric(series, errors="coerce").dropna().to_numpy(float)
    if values.size == 0:
        return np.nan
    return float(np.sqrt(np.mean(values**2)) * 1000.0)


rows = []
missing_or_short = []

for sat_code, sat_name in SATELLITES.items():
    stats_path = RESULTS_DIR / f"stats_{SOLUTION_TAG}_{sat_code}.csv"
    if not stats_path.exists():
        missing_or_short.append(f"{sat_code}: missing {stats_path.name}")
        continue

    stats = pd.read_csv(stats_path, parse_dates=["day"])
    n_days = int(stats["day"].nunique())
    if n_days != EXPECTED_DAYS:
        missing_or_short.append(f"{sat_code}: {n_days}/{EXPECTED_DAYS} days in {stats_path.name}")

    row = {
        "satellite": sat_code.upper(),
        "name": sat_name,
        "n_days": n_days,
    }
    for source_col, output_col in RMS_COLUMNS.items():
        row[output_col] = aggregate_rms_from_daily_rms(stats[source_col])
    rows.append(row)

monthly_rms = pd.DataFrame(rows)
monthly_rms

,satellite,name,n_days,R_rms_mm,T_rms_mm,N_rms_mm
0,SRL,SARAL,30,7.722173,20.064245,23.073580
1,CS2,CryoSat-2,28,9.028467,26.098021,24.877775
2,H2C,HY-2C,30,9.813042,28.542379,37.385234
3,H2D,HY-2D,28,8.403608,33.091994,28.194816
4,JA3,Jason-3,29,10.236997,28.213164,44.141265
5,S3A,Sentinel-3A,28,8.483001,23.408688,30.564143
6,S3B,Sentinel-3B,28,8.295338,23.342053,25.710673
7,S6A,Sentinel-6A,30,8.143326,28.775056,38.603948


In [10]:
if missing_or_short:
    print("Warning: not every satellite has 30 saved daily statistics:")
    for item in missing_or_short:
        print(f"- {item}")
else:
    print("All satellites have 30 saved daily statistics.")

- cs2: 28/30 days in stats_gop_vs_ssa_cs2.csv
- h2d: 28/30 days in stats_gop_vs_ssa_h2d.csv
- ja3: 29/30 days in stats_gop_vs_ssa_ja3.csv
- s3a: 28/30 days in stats_gop_vs_ssa_s3a.csv
- s3b: 28/30 days in stats_gop_vs_ssa_s3b.csv


In [11]:
from pathlib import Path
from doris.output.tables import Col, save_latex_table

table_df = monthly_rms[["satellite", "R_rms_mm", "T_rms_mm", "N_rms_mm"]].copy()

cols = [
    Col("satellite", "Družice", None),
    Col("R_rms_mm", "R [mm]", 2),
    Col("T_rms_mm", "T [mm]", 2),
    Col("N_rms_mm", "N [mm]", 2),
]

table_path = TABLES_DIR / "gop_vs_ssa_satellites_doy001-doy030_hermite_rms.tex"

latex = save_latex_table(
    table_df,
    table_path,
    cols=cols,
    caption="Souhrnné RMS rozdílů GOP--SSA v RTN za celé zpracované období",
    label="tab:gop_vs_ssa_satellites_doy001-doy030_hermite_rms",
    escape=False,
    index=False,
    position="H",
    centering=True,
    font_size="\\small",
    print_preview=True,
)

print(f"Saved: {table_path}")

\begin{table}[H]
\centering
\small
\caption{Souhrnné RMS rozdílů GOP--SSA v RTN za celé zpracované období}
\label{tab:gop_vs_ssa_satellites_doy001-doy030_hermite_rms}
\begin{tabular}{lrrr}
\toprule
Družice & R [mm] & T [mm] & N [mm] \\
\midrule
SRL & 7.72 & 20.06 & 23.07 \\
CS2 & 9.03 & 26.10 & 24.88 \\
H2C & 9.81 & 28.54 & 37.39 \\
H2D & 8.40 & 33.09 & 28.19 \\
JA3 & 10.24 & 28.21 & 44.14 \\
S3A & 8.48 & 23.41 & 30.56 \\
S3B & 8.30 & 23.34 & 25.71 \\
S6A & 8.14 & 28.78 & 38.60 \\
\bottomrule
\end{tabular}
\end{table}

Saved: C:\Users\michal\Desktop\MasterThesis-DorisAnalysis\LaTeX\tables\results\satellites\hermite\gop_vs_ssa_satellites_doy001-doy030_hermite_rms.tex
